# DDR: cohort, grouping, split, preprocessing and inference (lock v5.1)

Runs **after** `04_results/locked_config_v5_1_ddr.json` exists and refuses otherwise. It:

1. reassembles DDR from the chunks in `08_ddr` and checks the archive hash against intake;
2. applies the locked cohort (grades 0 to 4; byte-duplicate rule), builds the label-free filename groups, reports the fellow-eye assumption check, and writes the intraset split `01_splits/ddr_split_intraset_v2.csv` **before any model runs**;
3. preprocesses with the recovered v1 pipeline (RGB, 512x512 Lanczos, PNG) and saves the processed images to `08_ddr` for provenance;
4. runs all 33 frozen models and writes `03_predictions/ddr_predictions_<model>[_seedK]_v2.csv` (float64).

**No performance metric is computed or printed here** (lock v5.1 forbids it). Resumable: models whose prediction file exists are skipped.

Runtime: A100, High-RAM. Needs the `HF_TOKEN` secret for the RETFound config. About 1 to 1.5 hours, mostly preprocessing and RETFound.

In [2]:
# Cell 1: environment, lock guard
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install timm transformers
import os, io, re, glob, json, zipfile, hashlib, datetime, shutil, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

BASE    = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, CKPT, PREDS, RESULTS, AUDIT, DDR_IN = (f'{BASE}/{d}' for d in
    ('01_splits', '02_checkpoints', '03_predictions', '04_results', '99_audit', '08_ddr'))
LOCK_FILE = f'{RESULTS}/locked_config_v5_1_ddr.json'
LOCAL = '/content/ddr_local'; PNG = '/content/ddr_png'
for d in (LOCAL, PNG): os.makedirs(d, exist_ok=True)

def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(chunk), b''): h.update(blk)
    return h.hexdigest()

assert os.path.exists(LOCK_FILE), 'Lock v5.1 missing: do not run.'
LOCK = json.load(open(LOCK_FILE)); LOCK_SHA = sha256_file(LOCK_FILE)
existing = glob.glob(f'{PREDS}/ddr_predictions_*_v2.csv')
if existing:
    assert os.path.getmtime(LOCK_FILE) < min(os.path.getmtime(p) for p in existing), 'A DDR prediction predates the lock.'
    print(f'Resuming: {len(existing)} DDR prediction files already exist, all after the lock.')
MODELS = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
          'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
          'retfound_binary','retfound_5class','retfound_adversarial']
def tag(k): return '' if k == 0 else f'_seed{k}'
print('Lock v5.1', LOCK_SHA[:16], '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE', '| CPUs:', os.cpu_count())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lock v5.1 2531e1336f9487ee | GPU: NVIDIA A100-SXM4-40GB | CPUs: 12


In [3]:
# Cell 2: reassemble and extract DDR (hash must match intake)
chunks = sorted(glob.glob(f'{DDR_IN}/DDR-dataset.zip.0*')); assert len(chunks) == 10
ZIP = f'{LOCAL}/DDR-dataset.zip'
if not os.path.exists(f'{LOCAL}/DDR-dataset'):
    with open(ZIP, 'wb') as out:
        for c in chunks:
            with open(c, 'rb') as f:
                for blk in iter(lambda: f.read(1 << 24), b''): out.write(blk)
    assert sha256_file(ZIP) == LOCK['depends_on']['ddr_assembled_zip_sha256'], 'Archive differs from intake.'
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL)
    os.remove(ZIP)
print('DDR extracted.')

DDR extracted.


In [4]:
# Cell 3: cohort, grouping, assumption check, intraset split (written before any model runs)
MAN = f'{SPLITS}/ddr_manifest_v2.csv'
assert sha256_file(MAN) == LOCK['depends_on']['ddr_manifest_sha256'], 'Manifest changed since intake.'
man = pd.read_csv(MAN, dtype={'image_id': str})
man = man[man.grade <= 4].copy()
dups = man[man.sha256.duplicated(keep=False)]
drop_groups = dups.groupby('sha256').grade.nunique().loc[lambda s: s > 1].index
man = man[~man.sha256.isin(drop_groups)].drop_duplicates(subset='sha256', keep='first').reset_index(drop=True)
print(f'Cohort: {len(man):,} images | duplicate groups dropped for label conflict: {len(drop_groups)} | PDR: {(man.grade == 4).sum()}')

TS = re.compile(r'^\d{17}$'); NUM = re.compile(r'^(\d+)-(\d+)-(\d+)$')
def group_of(stem):
    if TS.match(stem): return f'ts_{stem[:8]}'
    m = NUM.match(stem)
    if m: return f'n_{m.group(1)}_{int(m.group(2)) // 50}'
    return f'single_{stem}'
man['group'] = man.image_id.map(group_of)
print('Groups:', man.group.nunique(), '| images per group, median', int(man.group.value_counts().median()),
      'max', int(man.group.value_counts().max()))

# Assumption check (reported, not decisive): do adjacent images agree in grade like fellow eyes do?
from sklearn.metrics import cohen_kappa_score
rng = np.random.default_rng(20260924)
def kappa_pairs(a, b): return round(float(cohen_kappa_score(a, b, weights='quadratic')), 3) if len(a) > 20 else None
num = man[man.image_id.str.match(NUM)].copy()
num['seq'] = num.image_id.str.extract(NUM)[1].astype(int); num['site'] = num.image_id.str.extract(NUM)[0]
num = num.sort_values(['site', 'seq']); nxt = num.shift(-1)
adj = (num.site == nxt.site) & (nxt.seq - num.seq == 1)
ts = man[man.image_id.str.match(TS)].copy()
ts['t'] = pd.to_datetime(ts.image_id.str[:14], format='%Y%m%d%H%M%S', errors='coerce'); ts = ts.dropna(subset=['t']).sort_values('t')
tn = ts.shift(-1); close = (tn.t - ts.t).dt.total_seconds().between(0, 120)
rand_a = rng.choice(man.grade.values, 3000); rand_b = rng.choice(man.grade.values, 3000)
CHECK = {'numbered_adjacent_pairs': int(adj.sum()),
         'numbered_adjacent_kappa': kappa_pairs(num.grade[adj].values, nxt.grade[adj].values.astype(int)),
         'timestamp_pairs_within_120s': int(close.sum()),
         'timestamp_pairs_kappa': kappa_pairs(ts.grade[close].values, tn.grade[close].values.astype(int)),
         'random_pairs_kappa': kappa_pairs(rand_a, rand_b)}
print('Fellow-eye assumption check:', CHECK)

groups = man.groupby('group').size().sample(frac=1.0, random_state=20260924)
half, cnt = {}, {'A': 0, 'B': 0}
for g, n in groups.items():
    h = 'A' if cnt['A'] <= cnt['B'] else 'B'; half[g] = h; cnt[h] += n
man['half'] = man.group.map(half)
SPLIT_OUT = f'{SPLITS}/ddr_split_intraset_v2.csv'
if os.path.exists(SPLIT_OUT):
    prev = pd.read_csv(SPLIT_OUT, dtype={'image_id': str}).set_index('image_id')
    assert (prev.loc[man.image_id, 'half'].values == man.half.values).all(), 'Split differs from the existing split file.'
else:
    man[['image_id','filename','grade','group','half','relpath','sha256']].to_csv(SPLIT_OUT, index=False)
print(man.groupby('half').agg(images=('image_id','size'), pdr=('grade', lambda g: int((g == 4).sum())),
                              referable=('grade', lambda g: round(float((g >= 2).mean()), 4))))

Cohort: 12,422 images | duplicate groups dropped for label conflict: 0 | PDR: 913
Groups: 332 | images per group, median 41 max 594
Fellow-eye assumption check: {'numbered_adjacent_pairs': 6584, 'numbered_adjacent_kappa': 0.892, 'timestamp_pairs_within_120s': 3128, 'timestamp_pairs_kappa': 0.981, 'random_pairs_kappa': -0.017}
      images  pdr  referable
half                        
A       6205  382     0.4237
B       6217  531     0.4821


In [5]:
# Cell 4: preprocess with the recovered v1 pipeline, save processed zips to Drive
from multiprocessing import Pool
def v1_preprocess(fp): return Image.open(fp).convert('RGB').resize((512, 512), Image.LANCZOS)
def _work(args):
    iid, rel = args; out = f'{PNG}/{iid}.png'
    if os.path.exists(out) and os.path.getsize(out) > 0: return iid, 'skip'
    try: v1_preprocess(f'{LOCAL}/{rel}').save(out); return iid, 'ok'
    except Exception as e: return iid, f'error: {e}'
jobs = list(zip(man.image_id, man.relpath)); t0 = time.time(); res = []
with Pool(os.cpu_count()) as pool:
    for k, r in enumerate(pool.imap_unordered(_work, jobs, chunksize=16), 1):
        res.append(r)
        if k % 2000 == 0: print(f'  {k}/{len(jobs)} ({time.time()-t0:.0f}s)')
errs = [r for r in res if r[1].startswith('error')]; assert not errs, errs[:5]
PROC_DIR = f'{DDR_IN}/processed_v2'; os.makedirs(PROC_DIR, exist_ok=True)
PROC_ZIPS = {}
for bi, chunk in enumerate(np.array_split(sorted(man.image_id), 4), 1):
    zp = f'/content/ddr_processed_batch{bi}_v2.zip'
    if not os.path.exists(f'{PROC_DIR}/{os.path.basename(zp)}'):
        with zipfile.ZipFile(zp, 'w', zipfile.ZIP_STORED) as zw:
            for iid in chunk: zw.write(f'{PNG}/{iid}.png', f'{iid}.png')
        shutil.copy(zp, PROC_DIR)
    PROC_ZIPS[os.path.basename(zp)] = sha256_file(f'{PROC_DIR}/{os.path.basename(zp)}')
print(f'Preprocessed {len(res)} images in {time.time()-t0:.0f}s; processed zips on Drive.')

  2000/12422 (61s)
  4000/12422 (115s)
  6000/12422 (167s)
  8000/12422 (221s)
  10000/12422 (274s)
  12000/12422 (327s)
Preprocessed 12422 images in 367s; processed zips on Drive.


In [6]:
# Cell 5: frozen models, strict loading, prediction (no metrics)
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoConfig, AutoModel
RETFOUND_REPO = 'iszt/RETFound_mae_meh'
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                              transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])
class ImgDS(Dataset):
    def __init__(self, ids): self.ids = list(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return TF_EVAL(Image.open(f'{PNG}/{self.ids[i]}.png').convert('RGB')), i
class RETFound(nn.Module):
    def __init__(self, nc):
        super().__init__(); cfg = AutoConfig.from_pretrained(RETFOUND_REPO)
        self.base = AutoModel.from_config(cfg, add_pooling_layer=False)
        self.fc_norm = nn.LayerNorm(cfg.hidden_size, eps=1e-6); self.classifier = nn.Linear(cfg.hidden_size, nc)
    def forward(self, x):
        h = self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:, 1:].mean(dim=1)))
def load_model(name, k, dev):
    path = f'{CKPT}/eyepacs_{name}{tag(k)}_v2.pt'
    assert os.path.exists(path), f'Missing checkpoint {path}'
    blob = torch.load(path, map_location='cpu', weights_only=False); cfg = blob['config']
    m = RETFound(cfg['nc']) if cfg['arch'] == 'retfound' else timm.create_model(cfg['arch'], pretrained=False, num_classes=cfg['nc'])
    m.load_state_dict(blob['state_dict'], strict=True)
    return m.to(dev).eval(), cfg, sha256_file(path)
@torch.no_grad()
def predict(model, dev, bs):
    ids = man.image_id.tolist(); probs = np.zeros(len(ids), dtype=np.float64)
    for x, idx in DataLoader(ImgDS(ids), batch_size=bs, shuffle=False, num_workers=min(8, os.cpu_count()), pin_memory=True):
        out = model(x.to(dev, non_blocking=True)).float()
        p = torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
        probs[idx.numpy()] = p.double().cpu().numpy()
    return probs

dev = 'cuda' if torch.cuda.is_available() else 'cpu'; RECORD = {}
todo = [(k, m) for k in (0, 1, 2) for m in MODELS if not os.path.exists(f'{PREDS}/ddr_predictions_{m}{tag(k)}_v2.csv')]
print(f'to run: {len(todo)} of 33')
for i, (k, name) in enumerate(todo, 1):
    t0 = time.time(); model, cfg, ck_sha = load_model(name, k, dev)
    pr = predict(model, dev, 32 if cfg['arch'] == 'retfound' else 128)
    out = f'{PREDS}/ddr_predictions_{name}{tag(k)}_v2.csv'
    pd.DataFrame({'image_id': man.image_id, 'predicted_probability': pr, 'model_name': name, 'replicate': k}
                 ).to_csv(out, index=False, float_format='%.17g')
    RECORD[f'{name}{tag(k)}'] = {'checkpoint_sha256': ck_sha, 'prediction_sha256': sha256_file(out),
                                 'mean_probability': round(float(pr.mean()), 4)}
    print(f'[{i}/{len(todo)}] {name}{tag(k)}  mean p {pr.mean():.3f}  {time.time()-t0:.0f}s')
    del model; torch.cuda.empty_cache()

to run: 33 of 33
[1/33] densenet121_binary  mean p 0.348  25s
[2/33] densenet121_5class  mean p 0.366  24s
[3/33] efficientnet_b3_binary  mean p 0.309  25s
[4/33] efficientnet_b3_5class  mean p 0.295  25s
[5/33] resnet50_binary  mean p 0.298  28s
[6/33] resnet50_5class  mean p 0.317  27s
[7/33] vit_base_binary  mean p 0.370  34s
[8/33] vit_base_5class  mean p 0.352  36s


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

[9/33] retfound_binary  mean p 0.308  122s
[10/33] retfound_5class  mean p 0.355  126s
[11/33] retfound_adversarial  mean p 0.307  121s
[12/33] densenet121_binary_seed1  mean p 0.369  25s
[13/33] densenet121_5class_seed1  mean p 0.358  24s
[14/33] efficientnet_b3_binary_seed1  mean p 0.324  27s
[15/33] efficientnet_b3_5class_seed1  mean p 0.320  24s
[16/33] resnet50_binary_seed1  mean p 0.315  26s
[17/33] resnet50_5class_seed1  mean p 0.294  27s
[18/33] vit_base_binary_seed1  mean p 0.385  34s
[19/33] vit_base_5class_seed1  mean p 0.261  34s
[20/33] retfound_binary_seed1  mean p 0.353  126s
[21/33] retfound_5class_seed1  mean p 0.342  128s
[22/33] retfound_adversarial_seed1  mean p 0.327  130s
[23/33] densenet121_binary_seed2  mean p 0.355  26s
[24/33] densenet121_5class_seed2  mean p 0.342  25s
[25/33] efficientnet_b3_binary_seed2  mean p 0.330  26s
[26/33] efficientnet_b3_5class_seed2  mean p 0.330  27s
[27/33] resnet50_binary_seed2  mean p 0.285  30s
[28/33] resnet50_5class_seed2  m

In [7]:
# Cell 6: provenance record and log
done = sorted(glob.glob(f'{PREDS}/ddr_predictions_*_v2.csv')); assert len(done) == 33, f'{len(done)} of 33 present'
rec = {'written_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'lock_v5_1_sha256': LOCK_SHA,
       'cohort_images': int(len(man)), 'pdr': int((man.grade == 4).sum()), 'groups': int(man.group.nunique()),
       'split_file_sha256': sha256_file(SPLIT_OUT), 'fellow_eye_check': CHECK, 'processed_zips_sha256': PROC_ZIPS,
       'prediction_sha256': {os.path.basename(p): sha256_file(p) for p in done}, 'models_run_this_session': RECORD,
       'note': 'No DDR performance metric computed. Analysis notebook written after this record, run once.'}
with open(f'{AUDIT}/ddr_inference_record_v2.json', 'w') as f: json.dump(rec, f, indent=2)
with open(f'{AUDIT}/RUN_LOG.md', 'a') as f:
    f.write(f'''\n\n## DDR inference ({rec['written_utc'][:10]})\n\nLock v5.1 {LOCK_SHA[:16]}...; cohort {len(man):,} images, '''
            f'''{rec['pdr']} PDR, {rec['groups']} filename groups, intraset split written before inference. 33 prediction files. '''
            f'''No metrics computed. Record 99_audit/ddr_inference_record_v2.json.\n''')
print(json.dumps({k: v for k, v in rec.items() if k not in ('prediction_sha256', 'models_run_this_session')}, indent=1))

{
 "written_utc": "2026-09-23T15:53:17.663694+00:00",
 "lock_v5_1_sha256": "2531e1336f9487ee074b6154cf2b44367ae7b3b5bcba644369cb1065bb346012",
 "cohort_images": 12422,
 "pdr": 913,
 "groups": 332,
 "split_file_sha256": "dad437fa3f74351a8852f18fa7c1b5fdc316998a11589e02e585d8ce14147804",
 "fellow_eye_check": {
  "numbered_adjacent_pairs": 6584,
  "numbered_adjacent_kappa": 0.892,
  "timestamp_pairs_within_120s": 3128,
  "timestamp_pairs_kappa": 0.981,
  "random_pairs_kappa": -0.017
 },
 "processed_zips_sha256": {
  "ddr_processed_batch1_v2.zip": "58b1ffadbdea24ad11d3bf81d0415d5eaff64d729c3e3c23233a56fc13622e50",
  "ddr_processed_batch2_v2.zip": "ca1a5f1b300dc705a65bb631dbbc1cee633612f39e6c1572590bb6b458a80b06",
  "ddr_processed_batch3_v2.zip": "c903d3292a562a299af414aedb078e783de533c71451bb2197bc6b9968faeaae",
  "ddr_processed_batch4_v2.zip": "a3bf4d7355067833446f9d0dd85cd59afe5b2709773aa5aebea763a5eefa71e7"
 },
 "note": "No DDR performance metric computed. Analysis notebook written afte

## After this notebook
Send back `99_audit/ddr_inference_record_v2.json` (it includes the fellow-eye check). Move this executed notebook to `99_audit/DDR_prep_inference_v2.ipynb`. The DDR analysis notebook is built from lock v5.1 after that, and runs once.